In [1]:
!pip install -q gradio pymupdf python-docx sentence-transformers scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import fitz
import docx
import gradio as gr
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

d:\Photos\Certificates\All Documents\Rooman Notes\AIML-Template\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3971.15it/s]


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s+#.]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [5]:
def extract_text_from_pdf(file_path):
    text = ""
    pdf = fitz.open(file_path)
    for page in pdf:
        text += page.get_text()
    return text


def extract_text_from_docx(file_path):
    document = docx.Document(file_path)
    return "\n".join([para.text for para in document.paragraphs])


def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        return extract_text_from_pdf(file_path)

    elif ext == ".docx":
        return extract_text_from_docx(file_path)

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    else:
        return ""

In [6]:
programming_languages = [
    "python", "java", "c", "c++", "javascript", "html", "css",
    "sql", "php", "ruby", "go", "kotlin", "swift"
]

skills = [
    "machine learning", "deep learning", "nlp", "natural language processing",
    "data analysis", "pandas", "numpy", "scikit-learn", "tensorflow",
    "pytorch", "fastapi", "flask", "django", "react", "node.js",
    "git", "docker", "aws", "mysql", "postgresql", "mongodb",
    "power bi", "tableau", "excel"
]

In [7]:
def extract_items(text, items):
    found = []
    for item in items:
        pattern = r"\b" + re.escape(item) + r"\b"
        if re.search(pattern, text):
            found.append(item)
    return found


def overlap_score(resume_items, jd_items):
    if len(jd_items) == 0:
        return 0
    return len(set(resume_items) & set(jd_items)) / len(set(jd_items))

In [8]:
def extract_experience_score(text):
    patterns = [
        r"(\d+)\+?\s*years",
        r"(\d+)\+?\s*year",
        r"(\d+)\+?\s*yrs"
    ]

    years = []

    for pattern in patterns:
        matches = re.findall(pattern, text)
        for match in matches:
            years.append(int(match))

    if not years:
        return 0.3

    max_years = max(years)

    if max_years >= 5:
        return 1.0
    elif max_years >= 3:
        return 0.8
    elif max_years >= 1:
        return 0.6
    else:
        return 0.3

In [9]:
def rank_bulk_resumes(resume_files, job_description):
    if not resume_files:
        return pd.DataFrame({"Error": ["Please upload at least one resume."]})

    if job_description.strip() == "":
        return pd.DataFrame({"Error": ["Please enter a job description."]})

    clean_jd = clean_text(job_description)

    jd_skills = extract_items(clean_jd, skills)
    jd_languages = extract_items(clean_jd, programming_languages)

    resume_data = []

    for file in resume_files:
        file_path = file.name
        resume_text = extract_text(file_path)

        if resume_text.strip() == "":
            continue

        clean_resume = clean_text(resume_text)

        resume_skills = extract_items(clean_resume, skills)
        resume_languages = extract_items(clean_resume, programming_languages)

        skill_score = overlap_score(resume_skills, jd_skills)
        language_score = overlap_score(resume_languages, jd_languages)
        experience_score = extract_experience_score(clean_resume)

        resume_data.append({
            "Resume Name": os.path.basename(file_path),
            "Resume Text": clean_resume,
            "Matched Skills": ", ".join(set(resume_skills) & set(jd_skills)),
            "Missing Skills": ", ".join(set(jd_skills) - set(resume_skills)),
            "Matched Languages": ", ".join(set(resume_languages) & set(jd_languages)),
            "Missing Languages": ", ".join(set(jd_languages) - set(resume_languages)),
            "Skill Score": skill_score,
            "Language Score": language_score,
            "Experience Score": experience_score
        })

    if len(resume_data) == 0:
        return pd.DataFrame({"Error": ["No valid resume text found."]})

    resume_texts = [item["Resume Text"] for item in resume_data]

    tfidf = TfidfVectorizer(stop_words="english")
    tfidf_matrix = tfidf.fit_transform(resume_texts + [clean_jd])

    resume_vectors = tfidf_matrix[:-1]
    jd_vector = tfidf_matrix[-1]

    keyword_scores = cosine_similarity(resume_vectors, jd_vector).flatten()

    resume_embeddings = model.encode(resume_texts, show_progress_bar=False)
    jd_embedding = model.encode([clean_jd])

    semantic_scores = cosine_similarity(resume_embeddings, jd_embedding).flatten()

    results = []

    for i, item in enumerate(resume_data):
        skill_score = item["Skill Score"]
        language_score = item["Language Score"]
        semantic_score = semantic_scores[i]
        keyword_score = keyword_scores[i]
        experience_score = item["Experience Score"]

        final_score = (
            0.25 * skill_score +
            0.35 * language_score +
            0.20 * semantic_score +
            0.10 * keyword_score +
            0.10 * experience_score
        )

        if final_score >= 0.85:
            recommendation = "Excellent Match"
        elif final_score >= 0.70:
            recommendation = "Good Match"
        elif final_score >= 0.50:
            recommendation = "Moderate Match"
        else:
            recommendation = "Low Match"

        results.append({
            "Resume Name": item["Resume Name"],
            "Final Score (%)": round(final_score * 100, 2),
            "Skill Score (%)": round(skill_score * 100, 2),
            "Language Score (%)": round(language_score * 100, 2),
            "Semantic Score (%)": round(semantic_score * 100, 2),
            "Keyword Score (%)": round(keyword_score * 100, 2),
            "Experience Score (%)": round(experience_score * 100, 2),

            "Matched Languages": item["Matched Languages"],
            "Missing Languages": item["Missing Languages"],
            "Recommendation": recommendation
        })

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(by="Final Score (%)", ascending=False)
    result_df.insert(0, "Rank", range(1, len(result_df) + 1))

    return result_df

In [10]:
custom_css = """
:root {
    --accent: #3b82f6;
    --accent-dark: #1e40af;
    --accent-light: #dbeafe;
    --bg-primary: #ffffff;
    --bg-secondary: #f8fafc;
    --bg-tertiary: #f1f5f9;
    --text-primary: #0f172a;
    --text-secondary: #475569;
    --text-muted: #64748b;
    --border-light: #e2e8f0;
    --border-medium: #cbd5e1;
    --shadow-sm: 0 1px 2px rgba(0, 0, 0, 0.05);
    --shadow-md: 0 4px 6px rgba(0, 0, 0, 0.07);
    --shadow-lg: 0 10px 25px rgba(0, 0, 0, 0.08);
    --shadow-xl: 0 20px 40px rgba(0, 0, 0, 0.1);
}

.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
    min-height: 100vh !important;
    background: linear-gradient(135deg, #ffffff 0%, #f8fafc 50%, #f1f5f9 100%) !important;
    color: var(--text-primary) !important;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif !important;
    padding: 40px 24px !important;
}

.gradio-container * {
    color: var(--text-primary);
}

/* Hero section - minimal & elegant */
#hero {
    position: relative;
    padding: 56px 48px;
    border-radius: 20px;
    background: var(--bg-primary);
    border: 1px solid var(--border-light);
    box-shadow: var(--shadow-xl);
    margin-bottom: 32px;
    overflow: hidden;
}

#hero::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    right: 0;
    height: 1px;
    background: linear-gradient(90deg, transparent, var(--accent), transparent);
    opacity: 0.5;
}

#badge {
    display: inline-block;
    padding: 8px 14px;
    border-radius: 8px;
    background: var(--accent-light);
    border: 1px solid var(--accent);
    color: var(--accent-dark) !important;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 0.5px;
    text-transform: uppercase;
    margin-bottom: 16px;
}

#hero-title {
    font-size: 42px;
    font-weight: 800;
    letter-spacing: -0.5px;
    margin-bottom: 12px;
    color: var(--text-primary) !important;
    line-height: 1.2;
}

#hero-subtitle {
    font-size: 16px;
    line-height: 1.6;
    color: var(--text-secondary) !important;
    max-width: 900px;
    margin: 0;
}

/* Metric grid - clean cards */
.metric-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
    gap: 16px;
    margin-top: 32px;
}

.metric-card {
    background: var(--bg-secondary);
    border: 1px solid var(--border-light);
    border-radius: 12px;
    padding: 20px;
    text-align: center;
    box-shadow: var(--shadow-sm);
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
}

.metric-card:hover {
    transform: translateY(-2px);
    box-shadow: var(--shadow-md);
    border-color: var(--accent);
}

.metric-value {
    font-size: 32px;
    font-weight: 800;
    color: var(--accent) !important;
    margin-bottom: 8px;
}

.metric-label {
    font-size: 13px;
    color: var(--text-secondary) !important;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.3px;
}

/* Panel styling - clean & minimal */
.panel {
    position: relative;
    background: var(--bg-primary);
    border: 1px solid var(--border-light);
    border-radius: 16px;
    padding: 28px;
    box-shadow: var(--shadow-lg);
}

.panel-title {
    font-size: 18px;
    font-weight: 700;
    color: var(--text-primary) !important;
    margin-bottom: 8px;
}

.panel-desc {
    color: var(--text-secondary) !important;
    font-size: 14px;
    margin-bottom: 20px;
    line-height: 1.5;
}

/* Input fields - clean design */
textarea,
input[type="text"],
input[type="email"],
input[type="number"],
select {
    background: var(--bg-secondary) !important;
    border: 1px solid var(--border-light) !important;
    color: var(--text-primary) !important;
    border-radius: 10px !important;
    font-size: 14px !important;
    padding: 12px 14px !important;
    transition: all 0.2s ease !important;
}

textarea:focus,
input:focus {
    border-color: var(--accent) !important;
    box-shadow: 0 0 0 3px rgba(59, 130, 246, 0.1) !important;
    outline: none !important;
}

textarea::placeholder,
input::placeholder {
    color: var(--text-muted) !important;
}

label {
    color: var(--text-primary) !important;
    font-weight: 600 !important;
    font-size: 14px !important;
}

span,
p {
    color: var(--text-secondary) !important;
}

/* File upload - clean dashed border */
[data-testid="file"] {
    background: var(--bg-secondary) !important;
    border: 2px dashed var(--border-medium) !important;
    border-radius: 12px !important;
    padding: 28px !important;
    transition: all 0.3s ease !important;
}

[data-testid="file"]:hover {
    border-color: var(--accent) !important;
    background: var(--accent-light) !important;
}

/* Buttons - modern & minimal */
button.primary {
    background: var(--accent) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    font-size: 14px !important;
    padding: 12px 24px !important;
    box-shadow: var(--shadow-md) !important;
    transition: all 0.25s ease !important;
    text-transform: none !important;
}

button.primary:hover {
    background: var(--accent-dark) !important;
    box-shadow: var(--shadow-lg) !important;
    transform: translateY(-1px);
}

button.primary:active {
    transform: translateY(0);
}

button {
    border-radius: 10px !important;
    font-weight: 600 !important;
    font-size: 14px !important;
    transition: all 0.25s ease !important;
}

button:not(.primary) {
    background: var(--bg-tertiary) !important;
    border: 1px solid var(--border-light) !important;
    color: var(--text-primary) !important;
}

button:not(.primary):hover {
    background: var(--border-light) !important;
}

/* Table styling */
.dataframe {
    border-radius: 12px !important;
    overflow: hidden !important;
    border: 1px solid var(--border-light) !important;
    box-shadow: var(--shadow-lg) !important;
}

table {
    background: var(--bg-primary) !important;
}

thead tr th {
    background: linear-gradient(135deg, #1e40af 0%, #3b82f6 100%) !important;
    color: white !important;
    font-weight: 700 !important;
    font-size: 13px !important;
    text-transform: uppercase !important;
    letter-spacing: 0.3px !important;
    border: none !important;
    padding: 16px 12px !important;
}

tbody tr td {
    background: var(--bg-primary) !important;
    color: var(--text-primary) !important;
    border: 1px solid var(--border-light) !important;
    padding: 14px 12px !important;
    font-size: 13px !important;
}

tbody tr:hover td {
    background: var(--bg-secondary) !important;
}

tbody tr:nth-child(even) td {
    background: var(--bg-secondary) !important;
}

tbody tr:nth-child(even):hover td {
    background: var(--bg-tertiary) !important;
}

/* Info strip - feature highlights */
#info-strip {
    margin-top: 32px;
    padding: 28px;
    border-radius: 16px;
    background: linear-gradient(135deg, var(--bg-primary) 0%, var(--bg-secondary) 100%);
    border: 1px solid var(--border-light);
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(140px, 1fr));
    gap: 16px;
    box-shadow: var(--shadow-lg);
}

.info-item {
    text-align: center;
    padding: 20px 16px;
    border-radius: 12px;
    background: var(--bg-secondary);
    border: 1px solid var(--border-light);
    transition: all 0.3s ease;
}

.info-item:hover {
    transform: translateY(-3px);
    box-shadow: var(--shadow-md);
    border-color: var(--accent);
}

.info-title {
    font-weight: 700;
    color: var(--accent) !important;
    font-size: 14px !important;
    margin-bottom: 6px;
}

.info-sub {
    font-size: 12px;
    color: var(--text-muted) !important;
    line-height: 1.4;
}

.footer {
    text-align: center;
    color: var(--text-muted) !important;
    font-size: 13px;
    margin-top: 32px;
    padding-top: 24px;
    border-top: 1px solid var(--border-light);
}

/* Scrollbar */
::-webkit-scrollbar {
    width: 8px;
    height: 8px;
}

::-webkit-scrollbar-track {
    background: var(--bg-secondary);
}

::-webkit-scrollbar-thumb {
    background: var(--accent);
    border-radius: 4px;
}

::-webkit-scrollbar-thumb:hover {
    background: var(--accent-dark);
}

/* Responsive */
@media (max-width: 1024px) {
    .metric-grid {
        grid-template-columns: repeat(2, 1fr);
    }
    
    #info-strip {
        grid-template-columns: repeat(2, 1fr);
    }
}

@media (max-width: 640px) {
    #hero {
        padding: 32px 24px;
    }
    
    #hero-title {
        font-size: 28px;
    }
    
    .metric-grid,
    #info-strip {
        grid-template-columns: 1fr;
    }
    
    .gradio-container {
        padding: 20px 16px !important;
    }
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div id="hero">
        <div id="badge">AI-Powered Recruitment</div>
        <div id="hero-title">Resume Screening & Ranking</div>
        <div id="hero-subtitle">
            Upload multiple resumes and automatically rank candidates against a job description.
            Our intelligent system analyzes skills, experience, and relevance to find the best matches.
        </div>

        <div class="metric-grid">
            <div class="metric-card">
                <div class="metric-value">35%</div>
                <div class="metric-label">Skill Match</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">25%</div>
                <div class="metric-label">Languages</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">20%</div>
                <div class="metric-label">Semantic Match</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">20%</div>
                <div class="metric-label">Keywords & Experience</div>
            </div>
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            with gr.Group(elem_classes="panel"):
                gr.HTML("""
                <div class="panel-title">📄 Upload Resumes</div>
                <div class="panel-desc">
                    Select PDF, DOCX, or TXT files. Multiple uploads supported.
                </div>
                """)

                resume_files = gr.File(
                    label="Student Resumes",
                    file_count="multiple",
                    file_types=[".pdf", ".docx", ".txt"]
                )

                gr.HTML("""
                <div class="panel-title" style="margin-top: 24px;">📋 Job Description</div>
                <div class="panel-desc">
                    Paste or type the target job description.
                </div>
                """)

                job_description = gr.Textbox(
                    label="Job Description",
                    lines=14,
                    placeholder="Example: We are looking for a Python developer with experience in FastAPI, Docker, and AWS..."
                )

                with gr.Row():
                    submit_btn = gr.Button(
                        "🚀 Analyze & Rank",
                        variant="primary",
                        scale=2
                    )
                    clear_btn = gr.ClearButton(
                        components=[resume_files, job_description],
                        value="↻ Reset",
                        scale=1
                    )

        with gr.Column(scale=2):
            with gr.Group(elem_classes="panel"):
                gr.HTML("""
                <div class="panel-title">🏆 Results</div>
                <div class="panel-desc">
                    Candidates ranked by relevance score. Higher scores indicate better matches.
                </div>
                """)

                output_table = gr.Dataframe(
                    label="Ranking Results",
                    wrap=True,
                    interactive=False
                )

    gr.HTML("""
    <div id="info-strip">
        <div class="info-item">
            <div class="info-title">Skills</div>
            <div class="info-sub">Technical & soft skills match</div>
        </div>
        <div class="info-item">
            <div class="info-title">Languages</div>
            <div class="info-sub">Programming language match</div>
        </div>
        <div class="info-item">
            <div class="info-title">Semantics</div>
            <div class="info-sub">AI-powered similarity</div>
        </div>
        <div class="info-item">
            <div class="info-title">Keywords</div>
            <div class="info-sub">Keyword relevance scoring</div>
        </div>
        <div class="info-item">
            <div class="info-title">Experience</div>
            <div class="info-sub">Years & roles analysis</div>
        </div>
    </div>

    <div class="footer">
        Built for intelligent resume screening and candidate evaluation
    </div>
    """)

    submit_btn.click(
        fn=rank_bulk_resumes,
        inputs=[resume_files, job_description],
        outputs=output_table
    )

demo.launch(share=True)

C:\Users\adity\AppData\Local\Temp\ipykernel_7696\3746959529.py:387: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
